# Simple RAG with LangChain (Python 3.11.9)

This notebook loads five realistic, fictional training PDFs from the `sample_pdfs` folder and uses them as a RAG knowledge base. Each PDF has three pages of document text.

**Flow:** load PDFs → split text → index chunks → retrieve relevant text → answer questions.

Keep this notebook beside the `sample_pdfs` folder and run cells from top to bottom. You need an OpenAI API key.

## 1. Install packages and set your API key

Run the install cell once. Restart the kernel if prompted. Add your key to a `.env` file beside this notebook:

```text
OPENAI_API_KEY=your-key-here
```

Keep your real key private.

In [1]:
%pip install -q  langchain-openai==1.7.2 langchain-core==1.6.13 openai==3.36.1 python-dotenv==1.2.4 pydantic==2.14.1 pandas==3.0.6 ipython==9.17.1 ipykernel==7.4.0 langchain==1.4.4 langchain-classic==1.0.8 langchain-community==0.4.2 langchain-chroma==1.1.0 langchain-text-splitters==1.1.3 chromadb==1.5.9 pypdf==6.8.0


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file or your environment, then rerun this cell.")

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 2. Put the PDF files beside the notebook

Place the provided `sample_pdfs` folder in the same directory as this notebook. It should contain `company_overview.pdf`, `leave_policy.pdf`, `expense_policy.pdf`, `security_policy.pdf`, and `remote_work_policy.pdf`. This notebook reads those files directly; it does not generate them.

## 3. Load all five PDF files

Each page becomes a document, with the PDF filename and page number kept as metadata.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_dir = Path("sample_pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))
if len(pdf_files) != 5:
    raise FileNotFoundError("Expected five PDF files in the sample_pdfs folder. Put that folder beside the notebook.")

pages = []
for pdf_file in pdf_files:
    pages.extend(PyPDFLoader(str(pdf_file)).load())

print("PDF files:", len(pdf_files))
print("Pages loaded:", len(pages))
for pdf_file in pdf_files:
    page_count = sum(1 for page in pages if Path(page.metadata["source"]).name == pdf_file.name)
    print(f"{pdf_file.name}: {page_count} pages")
    assert page_count >= 2, f"{pdf_file.name} should have at least two pages"


C:\Users\Admin\AppData\Local\Temp\ipykernel_11480\2218931891.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


PDF files: 5
Pages loaded: 15
company_overview.pdf: 3 pages
expense_policy.pdf: 3 pages
leave_policy.pdf: 3 pages
remote_work_policy.pdf: 3 pages
security_policy.pdf: 3 pages


## 4. Split documents into chunks

Chunking breaks each PDF into smaller pieces that can be searched.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print("Chunks created:", len(chunks))
print("Example chunk:", chunks[0].page_content[:300])

Chunks created: 45
Example chunk: Company Overview
1. Organization and Services
Northstar Services is a fictional professional services company used for training and demonstrations. It
provides technology consulting, application support, and process improvement services to medium and
large organizations. Its offices are located in B


## 5. Create embeddings and store chunks in Chroma

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_documents",
    persist_directory="chroma_db"
)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector database is ready.")

Vector database is ready.


## 6. Build the RAG question-answer chain

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If the answer is missing, say you could not find it.\n\nContext:\n{context}"),
    ("human", "{input}")
])
answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, answer_chain)

## 7. Ask questions about the PDFs

In [8]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source files:", sorted({Path(doc.metadata["source"]).name for doc in result["context"]}))
    print()
    return result

ask("How many paid annual leave days do full-time employees get?")

Question: How many paid annual leave days do full-time employees get?
Answer: Full-time employees receive 18 days of paid annual leave for each calendar year.
Source files: ['leave_policy.pdf']



{'input': 'How many paid annual leave days do full-time employees get?',
 'context': [Document(id='2fa8e3a3-c0c3-45dc-a31e-83ff1bf1d756', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'source': 'sample_pdfs\\leave_policy.pdf', 'subject': '(unspecified)', 'moddate': '2026-10-28T09:15:21+02:00', 'creator': '(unspecified)', 'total_pages': 3, 'keywords': '', 'author': 'Training example', 'page': 0, 'page_label': '1', 'creationdate': '2026-10-28T09:15:21+02:00', 'title': 'Leave Policy', 'trapped': '/False'}, page_content='Leave Policy\n1. Annual and Personal Leave\nIn this fictional training policy, full-time employees receive 18 days of paid annual leave for each\ncalendar year. Annual leave is intended for rest, personal commitments, and planned time away from\nwork. Part-time employees and employees who join or leave during the year should check their\nindividual entitlement with People Operations.\nEmployees should submit planned leave in the leave system at least five w

In [9]:
ask("When should employees submit expense claims?")
ask("What should I do if I receive a phishing message?")
ask("How many days per week can an employee work remotely?")

Question: When should employees submit expense claims?
Answer: Employees should submit expense claims in the expense system within 30 days of the purchase.
Source files: ['expense_policy.pdf']

Question: What should I do if I receive a phishing message?
Answer: If you receive a phishing message, do not click suspicious links, open unexpected attachments, or reply with sensitive information. Use the email client’s report function or forward the message to the IT help desk according to local instructions. Report it promptly so IT can protect data and arrange next steps.
Source files: ['remote_work_policy.pdf', 'security_policy.pdf']

Question: How many days per week can an employee work remotely?
Answer: An employee can work remotely up to two days per week with manager approval.
Source files: ['remote_work_policy.pdf']



{'input': 'How many days per week can an employee work remotely?',
 'context': [Document(id='a8df241c-643a-4a97-b7f0-298170f18098', metadata={'creationdate': '2026-10-28T09:15:21+02:00', 'producer': 'ReportLab PDF Library - (opensource)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-10-28T09:15:21+02:00', 'page_label': '1', 'source': 'sample_pdfs\\remote_work_policy.pdf', 'trapped': '/False', 'creator': '(unspecified)', 'total_pages': 3, 'page': 0, 'author': 'Training example', 'title': 'Remote Work Policy'}, page_content='Remote Work Policy\n1. Eligibility and Weekly Schedule\nThis fictional remote work policy allows eligible employees to work remotely up to two days per week\nwith manager approval. Remote work is an arrangement for performing normal duties away from the\noffice; it does not change role expectations, working hours, or service commitments.\nEmployees and managers should agree on a regular schedule that supports customer needs, team'),
  Document(id='019

## Summary

The separate PDF files are the knowledge sources. The loader reads them, the splitter creates searchable chunks, Chroma retrieves relevant text, and the chat model answers using those chunks. The PDFs are provided as files, so no document-content dictionary is needed in this notebook.